# NiyamTrace-X — External Multi-Benchmark / Multi-Model Validation

This notebook is **external-benchmark only**.

It does **not** rerun NiyamTrace-Bench3, Anchor Lock V3, or the paper-generation workflow.

## External suites

1. **BFCL V4** — official `bfcl generate` + `bfcl evaluate`.
2. **AgentDojo** — utility + prompt-injection security.
3. **AgentDyn** — dynamic/open-ended prompt-injection evaluation.
4. **τ³ / tau2-bench** — stateful multi-step agent evaluation.
5. **MLCL** — official-release gate; never reconstructed from the paper.
6. **MCP-SafetyBench** — optional, isolated real-MCP-server benchmark.

## Multi-model strategy

- **AgentDojo / AgentDyn / τ³** use an OpenRouter key by default with multiple model families.
- **BFCL** uses the model providers officially supported by BFCL. The notebook auto-enables models only when the corresponding provider key exists.
- **MCP-SafetyBench** stays opt-in because it can perform real external actions.

## Evidence rule

A benchmark receives a quantitative row only when the official runner/scorer produces a numeric benchmark-native result.
Installation success, repository cloning, or endpoint reachability are **not** counted as evidence.

In [ ]:
# ============================================================
# CELL 1 — CORE SETUP
# ============================================================
from pathlib import Path
from getpass import getpass
from datetime import datetime
import os, sys, json, re, time, random, hashlib, zipfile, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path("/content/NTX_EXTERNAL_BENCHMARKS") if Path("/content").exists() else Path.cwd()/"NTX_EXTERNAL_BENCHMARKS"
WORK = BASE/"work"
RESULTS = BASE/"results"
RAW = RESULTS/"raw"
for p in [BASE, WORK, RESULTS, RAW]:
    p.mkdir(parents=True, exist_ok=True)

MODE = os.getenv("NTX_RUN_MODE","QUICK").upper()
assert MODE in {"QUICK","STANDARD","FULL"}

LIMITS = {
    "QUICK":    {"bfcl_per_category":10,  "dojo_suites":["banking"], "agentdyn_suites":["shopping"], "tau_tasks":3},
    "STANDARD": {"bfcl_per_category":50,  "dojo_suites":["banking","workspace"], "agentdyn_suites":["shopping","github"], "tau_tasks":20},
    "FULL":     {"bfcl_per_category":None, "dojo_suites":["banking","slack","travel","workspace"], "agentdyn_suites":["shopping","github","dailylife"], "tau_tasks":None},
}[MODE]

def run(cmd, cwd=None, env=None, timeout=None, check=False):
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, errors="replace", timeout=timeout)
    if check and p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout)[-4000:])
    return p

def log(name, p):
    (RESULTS/name).write_text((p.stdout or "")+"\nSTDERR\n"+(p.stderr or ""))

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()

def ensure_uv():
    if not shutil.which("uv"):
        run([sys.executable,"-m","pip","install","-q","uv"], check=True)

def clone(url, dest):
    dest=Path(dest)
    if not dest.exists():
        p=run(["git","clone","--depth","1",url,str(dest)])
        if p.returncode: raise RuntimeError(p.stderr[-3000:])
    return run(["git","-C",str(dest),"rev-parse","HEAD"],check=True).stdout.strip()

print("Mode:", MODE)
print("Base:", BASE)
print("Python:", sys.version.split()[0])
print("Limits:", LIMITS)

## Credentials and model configuration

The model IDs are already configured.

You only provide the keys you actually have.

### Recommended minimum

- `OPENROUTER_API_KEY` — enables AgentDojo, AgentDyn and τ³ for open models.
- One or more BFCL-native provider keys:
  - `QWEN_API_KEY`
  - `GLM_API_KEY`
  - `OPENAI_API_KEY`
  - `GOOGLE_API_KEY`

The notebook never writes secret values to its manifest.

In [ ]:
# ============================================================
# CELL 2 — SECURE CREDENTIAL COLLECTION
# ============================================================
def secret(name, required=False):
    value=os.getenv(name,"").strip()
    if value: return value
    prompt=f"{name}"
    if not required: prompt += " (press Enter to skip)"
    value=getpass(prompt+": ").strip()
    if required and not value:
        raise RuntimeError(f"{name} is required.")
    if value:
        os.environ[name]=value
    return value

OPENROUTER_API_KEY = secret("OPENROUTER_API_KEY", required=False)

# Official BFCL provider keys — all optional.
QWEN_API_KEY   = secret("QWEN_API_KEY", required=False)
GLM_API_KEY    = secret("GLM_API_KEY", required=False)
OPENAI_API_KEY = secret("OPENAI_API_KEY", required=False)
GOOGLE_API_KEY = secret("GOOGLE_API_KEY", required=False)

# Optional service keys for MCP-SafetyBench.
SERP_API_KEY = os.getenv("SERP_API_KEY","").strip()
SERPER_API_KEY = os.getenv("SERPER_API_KEY","").strip()
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY","").strip()
GITHUB_PERSONAL_ACCESS_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN","").strip()

print("Credential availability:")
for name in ["OPENROUTER_API_KEY","QWEN_API_KEY","GLM_API_KEY","OPENAI_API_KEY","GOOGLE_API_KEY"]:
    print(f"  {name:<22}", "SET" if os.getenv(name,"").strip() else "MISSING")

In [ ]:
# ============================================================
# CELL 3 — PRECONFIGURED EXTERNAL MODEL MATRIX
# ============================================================

# AgentDojo/AgentDyn model enums are official benchmark enum names.
AGENT_MODELS = [
    {
        "label":"qwen3-235b",
        "family":"Qwen",
        "agentdojo_enum":"QWEN3_235B",
        "tau_model":"openrouter/qwen/qwen3-235b-a22b-2507",
        "requires":"OPENROUTER_API_KEY",
    },
    {
        "label":"llama-3.3-70b",
        "family":"Llama",
        "agentdojo_enum":"LLAMA_3_3_70B",
        "tau_model":"openrouter/meta-llama/llama-3.3-70b-instruct",
        "requires":"OPENROUTER_API_KEY",
    },
    {
        "label":"gpt-oss-120b",
        "family":"GPT-OSS",
        "agentdojo_enum":"",
        "tau_model":"openrouter/openai/gpt-oss-120b",
        "requires":"OPENROUTER_API_KEY",
    },
]

# BFCL names come from the official supported-model table.
BFCL_MODELS = [
    {"label":"bfcl-qwen3-235b","family":"Qwen","bfcl_model":"qwen3-235b-a22b-instruct-2507-FC","requires":"QWEN_API_KEY"},
    {"label":"bfcl-glm-4.6","family":"GLM","bfcl_model":"glm-4.6-FC","requires":"GLM_API_KEY"},
    {"label":"bfcl-gpt5-mini","family":"OpenAI","bfcl_model":"gpt-5-mini-2025-08-07-FC","requires":"OPENAI_API_KEY"},
    {"label":"bfcl-gemini-2.5-flash","family":"Gemini","bfcl_model":"gemini-2.5-flash-FC","requires":"GOOGLE_API_KEY"},
]

AGENT_MODELS_ACTIVE=[m for m in AGENT_MODELS if os.getenv(m["requires"],"").strip()]
BFCL_MODELS_ACTIVE=[m for m in BFCL_MODELS if os.getenv(m["requires"],"").strip()]

print("Agent benchmark models:", [(m["label"],m["family"]) for m in AGENT_MODELS_ACTIVE])
print("BFCL models:", [(m["bfcl_model"],m["family"]) for m in BFCL_MODELS_ACTIVE])

if not AGENT_MODELS_ACTIVE and not BFCL_MODELS_ACTIVE:
    raise RuntimeError("No usable external model credentials are configured.")

# Stage A — BFCL V4

The notebook uses the official Berkeley Function Calling Leaderboard workflow:

`bfcl generate` → `bfcl evaluate`

For QUICK/STANDARD modes it creates `test_case_ids_to_generate.json` from the official benchmark data so evaluation is explicitly partial and cheap enough to smoke-test.

FULL mode runs the selected categories without partial sampling.

In [ ]:
# ============================================================
# CELL 4 — INSTALL / PREPARE OFFICIAL BFCL
# ============================================================
BFCL_ROOT=WORK/"bfcl_project"
BFCL_ENV=WORK/"bfcl_py312"
BFCL_ROOT.mkdir(exist_ok=True)

bfcl_status=[]

if BFCL_MODELS_ACTIVE:
    ensure_uv()
    run(["uv","python","install","3.12"])
    if not BFCL_ENV.exists():
        run(["uv","venv",str(BFCL_ENV),"--python","3.12"],check=True)
    BP=str(BFCL_ENV/"bin"/"python")
    install=run([BP,"-m","pip","install","-q",
                 "numpy==1.26.4","pandas==2.2.3","soundfile","websockets",
                 "bfcl-eval"])
    log("bfcl_install.log",install)
    if install.returncode:
        raise RuntimeError("BFCL installation failed. See bfcl_install.log.")

    # BFCL_PROJECT_ROOT controls results, scores, .env, run-id file.
    env=os.environ.copy()
    env["BFCL_PROJECT_ROOT"]=str(BFCL_ROOT)

    # Write only into the temporary Colab workspace.
    env_lines=[
        f"QWEN_API_KEY={QWEN_API_KEY}",
        f"GLM_API_KEY={GLM_API_KEY}",
        f"OPENAI_API_KEY={OPENAI_API_KEY}",
        f"GOOGLE_API_KEY={GOOGLE_API_KEY}",
    ]
    (BFCL_ROOT/".env").write_text("\n".join(env_lines)+"\n")

    models=run([str(BFCL_ENV/"bin"/"bfcl"),"models"],env=env)
    log("bfcl_models.log",models)
    helpout=run([str(BFCL_ENV/"bin"/"bfcl"),"generate","--help"],env=env)
    log("bfcl_generate_help.log",helpout)

    print("BFCL environment ready.")
else:
    print("BFCL skipped: no BFCL-native provider keys.")

In [ ]:
# ============================================================
# CELL 5 — BUILD BFCL PARTIAL RUN IDS
# ============================================================
BFCL_CATEGORIES=["simple_python","multiple","parallel","parallel_multiple"]
if MODE!="QUICK":
    BFCL_CATEGORIES += ["irrelevance","live_simple","live_multiple"]

partial = LIMITS["bfcl_per_category"] is not None
run_ids={}

if BFCL_MODELS_ACTIVE and partial:
    # Locate packaged BFCL data files and sample real IDs.
    package_dir = Path(run([str(BFCL_ENV/"bin"/"python"),"-c",
                            "import bfcl_eval, pathlib; print(pathlib.Path(bfcl_eval.__path__[0]))"]).stdout.strip())
    for category in BFCL_CATEGORIES:
        candidates=list(package_dir.rglob(f"*{category}*.json"))
        # Prefer the actual benchmark-data file, not score/result/example files.
        candidates=[p for p in candidates if "data" in str(p).lower() and "score" not in str(p).lower()]
        ids=[]
        for p in candidates:
            try:
                lines=p.read_text(errors="ignore").splitlines()
                for line in lines:
                    try:
                        o=json.loads(line)
                        if isinstance(o,dict) and "id" in o:
                            ids.append(str(o["id"]))
                    except Exception:
                        pass
                if ids: break
            except Exception:
                pass
        if ids:
            run_ids[category]=ids[:LIMITS["bfcl_per_category"]]

    if not run_ids:
        raise RuntimeError("Could not locate BFCL benchmark IDs for partial evaluation.")
    (BFCL_ROOT/"test_case_ids_to_generate.json").write_text(json.dumps(run_ids,indent=2))
    print("BFCL partial IDs:", {k:len(v) for k,v in run_ids.items()})
else:
    print("BFCL full/category run; no partial ID file needed.")

In [ ]:
# ============================================================
# CELL 6 — RUN + SCORE BFCL
# ============================================================
bfcl_run_rows=[]
bfcl_score_rows=[]

if BFCL_MODELS_ACTIVE:
    env=os.environ.copy()
    env["BFCL_PROJECT_ROOT"]=str(BFCL_ROOT)

    BFCL_BIN=str(BFCL_ENV/"bin"/"bfcl")
    for m in BFCL_MODELS_ACTIVE:
        model=m["bfcl_model"]
        t=time.time()

        if partial:
            gen_cmd=[BFCL_BIN,"generate","--model",model,"--run-ids","--num-threads","1"]
        else:
            gen_cmd=[BFCL_BIN,"generate","--model",model,
                     "--test-category",",".join(BFCL_CATEGORIES),"--num-threads","1"]

        gp=run(gen_cmd,env=env)
        log(f"bfcl_generate_{m['label']}.log",gp)

        if gp.returncode==0:
            eval_cmd=[BFCL_BIN,"evaluate","--model",model]
            if not partial:
                eval_cmd += ["--test-category",",".join(BFCL_CATEGORIES)]
            else:
                eval_cmd += ["--partial-eval"]
            ep=run(eval_cmd,env=env)
            log(f"bfcl_evaluate_{m['label']}.log",ep)
        else:
            ep=None

        bfcl_run_rows.append({
            "benchmark":"BFCL-v4","model":m["label"],"family":m["family"],
            "generate_rc":gp.returncode,
            "evaluate_rc":None if ep is None else ep.returncode,
            "elapsed_s":time.time()-t,
            "status":"OK" if ep is not None and ep.returncode==0 else "ERROR"
        })

    # Parse official BFCL score outputs.
    score_root=BFCL_ROOT/"score"
    for csvp in score_root.rglob("*.csv"):
        try:
            df=pd.read_csv(csvp)
            df["source_file"]=str(csvp.relative_to(score_root))
            df.to_csv(RESULTS/f"bfcl_{csvp.name}",index=False)
        except Exception:
            pass

    # Parse per-category score JSONs; first row is BFCL metadata.
    for jp in score_root.rglob("*_score.json"):
        try:
            lines=jp.read_text(errors="ignore").splitlines()
            objs=[json.loads(x) for x in lines if x.strip()]
            if not objs: continue
            meta=objs[0]
            if isinstance(meta,dict) and "accuracy" in meta:
                # model name is first directory under score root
                rel=jp.relative_to(score_root)
                model_dir=rel.parts[0] if rel.parts else ""
                bfcl_score_rows.append({
                    "benchmark":"BFCL-v4",
                    "model":model_dir,
                    "category":jp.stem.replace("_score",""),
                    "metric":"accuracy",
                    "score":float(meta["accuracy"]),
                    "n":int(meta.get("total_count",0)),
                    "correct":int(meta.get("correct_count",0)),
                    "source_file":str(rel)
                })
        except Exception:
            pass

pd.DataFrame(bfcl_run_rows).to_csv(RESULTS/"bfcl_run_status.csv",index=False)
pd.DataFrame(bfcl_score_rows).to_csv(RESULTS/"bfcl_category_scores.csv",index=False)

display(pd.DataFrame(bfcl_run_rows))
display(pd.DataFrame(bfcl_score_rows).head(30))

# Stage B — AgentDojo

AgentDojo reports:

- average utility,
- injection-task success as user tasks,
- average security.

The notebook runs the official benchmark script with the `important_instructions` attack and parses those benchmark-native printed metrics.

In [ ]:
# ============================================================
# CELL 7 — AGENTDOJO INSTALL + RUN
# ============================================================
DOJO=WORK/"agentdojo"
dojo_rows=[]

if AGENT_MODELS_ACTIVE:
    commit=clone("https://github.com/ethz-spylab/agentdojo.git",DOJO)
    ensure_uv()
    sp=run(["uv","sync"],cwd=DOJO)
    log("agentdojo_setup.log",sp)

    for m in AGENT_MODELS_ACTIVE:
        enum=m["agentdojo_enum"]
        if not enum:
            dojo_rows.append({"benchmark":"AgentDojo","model":m["label"],"family":m["family"],"status":"UNSUPPORTED_ENUM"})
            continue

        for suite in LIMITS["dojo_suites"]:
            outdir=RAW/"agentdojo"/m["label"]/suite
            outdir.mkdir(parents=True,exist_ok=True)
            env=os.environ.copy()
            env["OPENROUTER_API_KEY"]=OPENROUTER_API_KEY

            cmd=["uv","run","python","-m","agentdojo.scripts.benchmark",
                 "-s",suite,"--model",enum,
                 "--attack","important_instructions",
                 "--logdir",str(outdir),
                 "--max-workers","1"]
            p=run(cmd,cwd=DOJO,env=env)
            log(f"agentdojo_{m['label']}_{suite}.log",p)

            txt=(p.stdout or "")+"\n"+(p.stderr or "")
            util=re.findall(r"Average utility:\s*([0-9.]+)%",txt)
            sec=re.findall(r"Average security:\s*([0-9.]+)%",txt)
            inj=re.findall(r"Passed injection tasks as user tasks:\s*(\d+)/(\d+)",txt)

            dojo_rows.append({
                "benchmark":"AgentDojo","model":m["label"],"family":m["family"],"suite":suite,
                "utility":float(util[-1])/100 if util else np.nan,
                "security":float(sec[-1])/100 if sec else np.nan,
                "injection_task_passed":int(inj[-1][0]) if inj else np.nan,
                "injection_task_total":int(inj[-1][1]) if inj else np.nan,
                "status":"OK" if p.returncode==0 and (util or sec) else "ERROR",
                "returncode":p.returncode,"commit":commit
            })

dojo=pd.DataFrame(dojo_rows)
dojo.to_csv(RESULTS/"agentdojo_metrics.csv",index=False)
display(dojo)

# Stage C — AgentDyn

AgentDyn uses the AgentDojo evaluation script and supports the `shopping`, `github`, and `dailylife` dynamic suites.

The same benchmark-native utility/security parser is used.

In [ ]:
# ============================================================
# CELL 8 — AGENTDYN INSTALL + RUN
# ============================================================
AD=WORK/"AgentDyn"
agentdyn_rows=[]

if AGENT_MODELS_ACTIVE:
    commit=clone("https://github.com/SaFo-Lab/AgentDyn.git",AD)
    ip=run([sys.executable,"-m","pip","install","-q","-e",str(AD)])
    log("agentdyn_install.log",ip)

    for m in AGENT_MODELS_ACTIVE:
        enum=m["agentdojo_enum"]
        if not enum:
            agentdyn_rows.append({"benchmark":"AgentDyn","model":m["label"],"family":m["family"],"status":"UNSUPPORTED_ENUM"})
            continue

        for suite in LIMITS["agentdyn_suites"]:
            outdir=RAW/"agentdyn"/m["label"]/suite
            outdir.mkdir(parents=True,exist_ok=True)
            env=os.environ.copy()
            env["OPENROUTER_API_KEY"]=OPENROUTER_API_KEY

            cmd=[sys.executable,"-m","agentdojo.scripts.benchmark",
                 "-s",suite,"--model",enum,
                 "--attack","important_instructions",
                 "--logdir",str(outdir),
                 "--max-workers","1"]
            p=run(cmd,cwd=AD,env=env)
            log(f"agentdyn_{m['label']}_{suite}.log",p)

            txt=(p.stdout or "")+"\n"+(p.stderr or "")
            util=re.findall(r"Average utility:\s*([0-9.]+)%",txt)
            sec=re.findall(r"Average security:\s*([0-9.]+)%",txt)
            inj=re.findall(r"Passed injection tasks as user tasks:\s*(\d+)/(\d+)",txt)

            agentdyn_rows.append({
                "benchmark":"AgentDyn","model":m["label"],"family":m["family"],"suite":suite,
                "utility":float(util[-1])/100 if util else np.nan,
                "security":float(sec[-1])/100 if sec else np.nan,
                "injection_task_passed":int(inj[-1][0]) if inj else np.nan,
                "injection_task_total":int(inj[-1][1]) if inj else np.nan,
                "status":"OK" if p.returncode==0 and (util or sec) else "ERROR",
                "returncode":p.returncode,"commit":commit
            })

agentdyn=pd.DataFrame(agentdyn_rows)
agentdyn.to_csv(RESULTS/"agentdyn_metrics.csv",index=False)
display(agentdyn)

# Stage D — τ³ / tau2-bench

τ³ uses LiteLLM, so the OpenRouter models are passed directly.

The notebook stores the official simulation JSONs and extracts numeric reward/success scalars per produced trajectory without mixing whole files into one “trajectory”.

In [ ]:
# ============================================================
# CELL 9 — τ³ INSTALL + RUN
# ============================================================
TAU=WORK/"tau2-bench"
tau_run_rows=[]

if AGENT_MODELS_ACTIVE:
    commit=clone("https://github.com/sierra-research/tau2-bench.git",TAU)
    ensure_uv()
    sp=run(["uv","sync"],cwd=TAU)
    log("tau_setup.log",sp)
    run(["uv","pip","install","websockets","soundfile"],cwd=TAU)

    domains=["airline","retail","telecom"]
    if MODE=="FULL":
        domains.append("banking_knowledge")

    for m in AGENT_MODELS_ACTIVE:
        llm=m["tau_model"]
        for domain in domains:
            tag=f"ntx_external_{m['label']}_{domain}_{int(time.time())}"
            env=os.environ.copy()
            env["OPENROUTER_API_KEY"]=OPENROUTER_API_KEY

            cmd=["uv","run","tau2","run",
                 "--domain",domain,
                 "--agent-llm",llm,
                 "--user-llm",llm,
                 "--num-trials","1",
                 "--max-concurrency","1",
                 "--seed",str(SEED),
                 "--save-to",tag]
            if LIMITS["tau_tasks"] is not None:
                cmd += ["--num-tasks",str(LIMITS["tau_tasks"])]

            p=run(cmd,cwd=TAU,env=env)
            log(f"tau_{m['label']}_{domain}.log",p)
            tau_run_rows.append({
                "benchmark":"tau3","model":m["label"],"family":m["family"],"domain":domain,
                "status":"OK" if p.returncode==0 else "ERROR",
                "returncode":p.returncode,"tag":tag,"commit":commit
            })

tau_runs=pd.DataFrame(tau_run_rows)
tau_runs.to_csv(RESULTS/"tau_run_status.csv",index=False)
display(tau_runs)

In [ ]:
# ============================================================
# CELL 10 — τ³ PER-TRAJECTORY RESULT NORMALIZATION
# ============================================================
tau_traj=[]

def find_numeric(obj, keys):
    if isinstance(obj,dict):
        for k in keys:
            if k in obj and isinstance(obj[k],(int,float,bool)):
                return float(obj[k])
        for v in obj.values():
            x=find_numeric(v,keys)
            if x is not None: return x
    elif isinstance(obj,list):
        for v in obj:
            x=find_numeric(v,keys)
            if x is not None:return x
    return None

if len(tau_runs):
    simroot=TAU/"data"/"tau2"/"simulations"
    for _,rr in tau_runs[tau_runs.status=="OK"].iterrows():
        tag=str(rr.tag)
        files=list(simroot.rglob(f"*{tag}*.json")) if simroot.exists() else []
        for p in files:
            try:
                obj=json.loads(p.read_text())
            except Exception:
                continue

            # Normalize root/list/container into individual simulations.
            items=[]
            if isinstance(obj,list):
                items=obj
            elif isinstance(obj,dict):
                for key in ["simulations","results","trajectories","data"]:
                    if isinstance(obj.get(key),list):
                        items=obj[key];break
                if not items:
                    items=[obj]

            for idx,item in enumerate(items):
                if not isinstance(item,dict):continue
                reward=find_numeric(item,["reward","score","success","task_success"])
                tau_traj.append({
                    "benchmark":"tau3","model":rr.model,"family":rr.family,"domain":rr.domain,
                    "trajectory_index":idx,"reward":reward,
                    "source_file":str(p.relative_to(TAU))
                })

tau_df=pd.DataFrame(tau_traj)
tau_df.to_csv(RESULTS/"tau_trajectory_metrics.csv",index=False)

tau_summary=(tau_df.groupby(["benchmark","model","family","domain"],dropna=False)
             .agg(n=("reward","count"),mean_reward=("reward","mean"))
             .reset_index()) if len(tau_df) else pd.DataFrame()

tau_summary.to_csv(RESULTS/"tau_summary.csv",index=False)
display(tau_summary)

# Stage E — MLCL official-release gate

The ACL 2026 paper is used as motivation, but this notebook **does not recreate MLCL from prose**.

Set `MLCL_OFFICIAL_PATH` only when you have an official benchmark release.

In [ ]:
# ============================================================
# CELL 11 — MLCL OFFICIAL RELEASE GATE
# ============================================================
mlcl_path=os.getenv("MLCL_OFFICIAL_PATH","").strip()
if mlcl_path and Path(mlcl_path).exists():
    mlcl_status={"benchmark":"MLCL","status":"OFFICIAL_SOURCE_PRESENT","path":mlcl_path,
                 "note":"Source present; add official runner command when released/documented."}
else:
    mlcl_status={"benchmark":"MLCL","status":"UNAVAILABLE_NO_OFFICIAL_SOURCE","path":"",
                 "note":"Not reconstructed from paper text."}

pd.DataFrame([mlcl_status]).to_csv(RESULTS/"mlcl_status.csv",index=False)
display(pd.DataFrame([mlcl_status]))

# Stage F — MCP-SafetyBench (optional / isolated)

This benchmark can perform real operations on external systems.

It is therefore **disabled by default**.

To enable it, explicitly set:

```python
ENABLE_MCP_SAFETYBENCH = True
```

Use a disposable environment/account and minimum-scope credentials.

In [ ]:
# ============================================================
# CELL 12 — MCP-SAFETYBENCH SAFE GATE
# ============================================================
ENABLE_MCP_SAFETYBENCH = False   # change to True only after reading the warning above

mcp_rows=[]

if ENABLE_MCP_SAFETYBENCH:
    MCP=WORK/"MCPSafety"
    commit=clone("https://github.com/xjzzzzzzzz/MCPSafety.git",MCP)
    ip=run([sys.executable,"-m","pip","install","-q","-r",str(MCP/"requirements.txt")])
    log("mcp_install.log",ip)

    # Core benchmark currently documents OpenAI/Anthropic/Gemini providers.
    provider_available=bool(OPENAI_API_KEY or GOOGLE_API_KEY)
    if not provider_available:
        mcp_rows.append({"benchmark":"MCP-SafetyBench","domain":"ALL","status":"SKIPPED_NO_SUPPORTED_LLM_PROVIDER_KEY","commit":commit})
    else:
        # Financial-analysis is attempted first because it does not require the
        # extra search/maps/GitHub credentials listed by the benchmark README.
        domains=[("financial_analysis","tests/benchmark/test_benchmark_financial_analysis.py")]
        for domain,script in domains:
            env=os.environ.copy()
            p=run([sys.executable,script],cwd=MCP,env=env)
            log(f"mcp_{domain}.log",p)
            mcp_rows.append({"benchmark":"MCP-SafetyBench","domain":domain,
                             "status":"OK" if p.returncode==0 else "ERROR",
                             "returncode":p.returncode,"commit":commit})
else:
    mcp_rows=[{"benchmark":"MCP-SafetyBench","domain":"ALL","status":"DISABLED_BY_DEFAULT_FOR_SAFETY"}]

pd.DataFrame(mcp_rows).to_csv(RESULTS/"mcp_run_status.csv",index=False)
display(pd.DataFrame(mcp_rows))

# Unified external benchmark table

Metrics remain benchmark-native.

The notebook intentionally does **not** average BFCL accuracy, AgentDojo security, AgentDyn utility, and τ³ reward into one meaningless global score.

In [ ]:
# ============================================================
# CELL 13 — UNIFIED BENCHMARK-NATIVE EVIDENCE TABLE
# ============================================================
rows=[]

# BFCL
for r in bfcl_score_rows:
    rows.append({
        "benchmark":"BFCL-v4","model":r["model"],"family":"",
        "slice":r["category"],"metric":"accuracy","score":r["score"],"n":r["n"],
        "paper_eligible":True
    })

# AgentDojo
if 'dojo' in globals() and len(dojo):
    for _,r in dojo[dojo.status=="OK"].iterrows():
        if pd.notna(r.utility):
            rows.append({"benchmark":"AgentDojo","model":r.model,"family":r.family,"slice":r.suite,
                         "metric":"utility","score":r.utility,"n":np.nan,"paper_eligible":True})
        if pd.notna(r.security):
            rows.append({"benchmark":"AgentDojo","model":r.model,"family":r.family,"slice":r.suite,
                         "metric":"security","score":r.security,"n":np.nan,"paper_eligible":True})

# AgentDyn
if 'agentdyn' in globals() and len(agentdyn):
    for _,r in agentdyn[agentdyn.status=="OK"].iterrows():
        if pd.notna(r.utility):
            rows.append({"benchmark":"AgentDyn","model":r.model,"family":r.family,"slice":r.suite,
                         "metric":"utility","score":r.utility,"n":np.nan,"paper_eligible":True})
        if pd.notna(r.security):
            rows.append({"benchmark":"AgentDyn","model":r.model,"family":r.family,"slice":r.suite,
                         "metric":"security","score":r.security,"n":np.nan,"paper_eligible":True})

# tau3
if 'tau_summary' in globals() and len(tau_summary):
    for _,r in tau_summary.iterrows():
        if pd.notna(r.mean_reward):
            rows.append({"benchmark":"tau3","model":r.model,"family":r.family,"slice":r.domain,
                         "metric":"mean_reward","score":r.mean_reward,"n":r.n,"paper_eligible":True})

unified=pd.DataFrame(rows)
unified.to_csv(RESULTS/"external_unified_evidence.csv",index=False)
display(unified)

In [ ]:
# ============================================================
# CELL 14 — CROSS-MODEL / CROSS-BENCHMARK SUMMARY
# ============================================================
summary=[]
if len(unified):
    for (b,m,metric),g in unified.groupby(["benchmark","model","metric"]):
        vals=g.score.dropna().astype(float)
        summary.append({
            "benchmark":b,"model":m,"metric":metric,
            "n_slices":len(vals),
            "mean_score":vals.mean() if len(vals) else np.nan,
            "min_score":vals.min() if len(vals) else np.nan,
            "max_score":vals.max() if len(vals) else np.nan,
        })
summary=pd.DataFrame(summary)
summary.to_csv(RESULTS/"external_model_summary.csv",index=False)
display(summary)

# Coverage matrix
if len(unified):
    coverage=(unified.assign(present=1)
              .pivot_table(index="model",columns="benchmark",values="present",aggfunc="max",fill_value=0))
    coverage.to_csv(RESULTS/"external_coverage_matrix.csv")
    display(coverage)
else:
    coverage=pd.DataFrame()

In [ ]:
# ============================================================
# CELL 15 — PAPER-READY FIGURES
# ============================================================
# One chart per benchmark/metric — no heterogeneous metric pooling.
if len(summary):
    for (bench,metric),g in summary.groupby(["benchmark","metric"]):
        fig,ax=plt.subplots(figsize=(8,max(3,0.5*len(g)+1)))
        gg=g.sort_values("mean_score")
        ax.barh(gg.model,gg.mean_score)
        ax.set_xlabel(metric.replace("_"," "))
        ax.set_title(f"{bench}: {metric}")
        fig.tight_layout()
        out=RESULTS/f"figure_{re.sub('[^A-Za-z0-9]+','_',bench)}_{re.sub('[^A-Za-z0-9]+','_',metric)}.png"
        fig.savefig(out,dpi=240,bbox_inches="tight")
        plt.show()

In [ ]:
# ============================================================
# CELL 16 — EXTERNAL CLAIM CHECKLIST
# ============================================================
checks=[]

def has_metric(bench):
    return bool(len(unified[unified.benchmark==bench])) if len(unified) else False

checks += [
    {"claim":"BFCL external function-calling evidence","status":"SUPPORTED" if has_metric("BFCL-v4") else "MISSING"},
    {"claim":"AgentDojo prompt-injection evidence","status":"SUPPORTED" if has_metric("AgentDojo") else "MISSING"},
    {"claim":"AgentDyn dynamic prompt-injection evidence","status":"SUPPORTED" if has_metric("AgentDyn") else "MISSING"},
    {"claim":"tau3 stateful external evidence","status":"SUPPORTED" if has_metric("tau3") else "MISSING"},
    {"claim":"MLCL external multilingual evidence","status":"MISSING" if mlcl_status["status"]!="OFFICIAL_SOURCE_PRESENT" else "SOURCE_PRESENT"},
    {"claim":"MCP-SafetyBench evidence","status":"RUN_ONLY" if any(r.get("status")=="OK" for r in mcp_rows) else "MISSING"},
]

covered_benchmarks=set(unified.benchmark.astype(str)) if len(unified) else set()
families=set(unified.family.dropna().astype(str)) - {""} if len(unified) else set()

checks += [
    {"claim":">=3 external benchmark families","status":"SUPPORTED" if len(covered_benchmarks)>=3 else "MISSING"},
    {"claim":">=3 external model families","status":"SUPPORTED" if len(families)>=3 else "MISSING"},
]

claims=pd.DataFrame(checks)
claims.to_csv(RESULTS/"external_claim_checklist.csv",index=False)
display(claims)

In [ ]:
# ============================================================
# CELL 17 — REPRODUCIBILITY MANIFEST + FINAL ZIP DOWNLOAD
# ============================================================
manifest={
    "experiment":"NTX-EXTERNAL-MULTI-BENCHMARK",
    "created_at":datetime.now().isoformat(),
    "mode":MODE,
    "seed":SEED,
    "agent_models":[{k:v for k,v in m.items() if k!="requires"} for m in AGENT_MODELS_ACTIVE],
    "bfcl_models":[{k:v for k,v in m.items() if k!="requires"} for m in BFCL_MODELS_ACTIVE],
    "claim_checklist":claims.to_dict("records"),
}

hashes={}
for p in sorted(RESULTS.rglob("*")):
    if p.is_file() and p.name not in {"FINAL_MANIFEST.json","SHA256SUMS.txt"}:
        hashes[str(p.relative_to(RESULTS))]=sha256_file(p)
manifest["artifact_sha256"]=hashes

(RESULTS/"FINAL_MANIFEST.json").write_text(json.dumps(manifest,indent=2,default=str))
(RESULTS/"SHA256SUMS.txt").write_text("\n".join(f"{h}  {k}" for k,h in sorted(hashes.items()))+"\n")

zipout=BASE/"NTX_EXTERNAL_MULTI_BENCHMARK_RESULTS.zip"
if zipout.exists():zipout.unlink()
with zipfile.ZipFile(zipout,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        if p.is_file():z.write(p,arcname=str(p.relative_to(RESULTS)))

print("Results ZIP:",zipout)
print("SHA-256:",sha256_file(zipout))
print("Size MiB:",round(zipout.stat().st_size/1024**2,3))
display(claims)

try:
    from google.colab import files
    files.download(str(zipout))
except Exception:
    pass